# M5: CLV 경제표현 + 양성 구매금액 가중학습 (Dunnhumby)

M2는 사용자의 축소추정 4구간 구매금액 분포와 degree-conditioned V rank, 상품의 전체·카테고리 내 대표 구매금액 위치를 4차원 경제표현으로 공동학습합니다. M4'는 hard negative 대신 CLV 수준에 따라 구매금액이 큰 양성 상품의 BPR 손실을 더 크게 학습합니다. A~F 여섯 arm을 같은 historical development 구간에서 비교하며 final test와 holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'd843761801773517e2021c29cafb367ba5bf4b81'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_economic_positive_weight import (
    configure_m5_economic_positive_run,
    preflight_summary,
    run_m5_economic_positive_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_economic_positive_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_economic_positive_weighting_historical_screen_v1',
    baseline_result_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1',
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_economic_positive_screen(cfg)

In [ ]:
from IPython.display import display

print('1) 절대지표: M1, M2, M4-prime, M5, 공동순열, degree gate')
display(result_df)
print('2) 대조군별 전체 성과 비교')
display(result_df.attrs['comparison'])
print('3) M2 x M4-prime 상호작용')
display(result_df.attrs['interaction'])
print('4) 실제 점수 영향력')
display(result_df.attrs['score_diagnostics'])
print('5) Top-10 변경')
display(result_df.attrs['top10_overlap'])
print('6) V 및 degree 사분위별 추천상품 평균 구매금액 백분위')
display(result_df.attrs['economic_recommendations'])
print('7) 사전 판정')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('8) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))